# 7조 실험 노트북 — E8 기준선 + 내 파생변수 추가

**실험 ID:** (예: E9) · **담당:** (이름) · **가설:** (한 줄) · **결과:** (CV 점수, 끝나면 기입)

사용법
1. 위 4칸을 먼저 채우고 `logs/experiments.md`에도 같은 줄을 추가한다 (중복 방지).
2. PART 0~2는 그대로 실행 (기준선 0.429 재현).
3. PART 3의 `my_features()`에 **내 파생변수 하나**만 넣고 PART 4 실행 → 점수 비교.
4. 결과를 맨 위와 `logs/experiments.md`에 적고 커밋.

규칙: test는 PART 5(제출)에서만 읽는다. `fit`은 학습 fold에서만. 한 번에 한 가지만 바꾼다.

## PART 0 · 환경

In [ ]:
# Colab 환경 준비: 팀 레포(src, refs)와 데이터 경로


# 구글 드라이브 연결 — 데이터(train.csv)는 드라이브에 두고 읽는다
from google.colab import drive
drive.mount('/content/drive')

# 팀 레포를 받아 src/features.py 와 refs/ 유전자 목록을 쓴다 (코드 복사 금지, import로 사용)
!git clone -q https://github.com/ten-infosec/dacon-cancer-subtype-oz-team7.git /content/repo
import sys; sys.path.insert(0, '/content/repo/src')

# 데이터 폴더 — 본인 드라이브 경로로 수정. 최종 제출본에서는 규정대로 '/data'로 바꾼다
DATA_DIR = '/content/drive/MyDrive/dacon_cancer/data'

# 모든 무작위 과정을 고정하는 시드 — 팀 공통 42
SEED = 42


# → 드라이브를 연결하고 팀 레포를 받아 공통 전처리 함수를 불러올 준비를 하겠다.

In [ ]:
# 라이브러리와 버전 기록 (제출 규정: 개발 환경·라이브러리 버전 기재)


import platform, sklearn, xgboost, numpy as np, pandas as pd
import features as F
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

print('OS', platform.platform())
print('python', platform.python_version(), '| pandas', pd.__version__, '| numpy', np.__version__,
      '| sklearn', sklearn.__version__, '| xgboost', xgboost.__version__)


# → 재현 검증을 위해 환경 버전을 출력해 두겠다.

## PART 1 · 데이터 (train만)

In [ ]:
# train 읽기와 기본 확인 — test.csv는 여기서 열지 않는다


train = pd.read_csv(f'{DATA_DIR}/train.csv')
genes = F.gene_columns(train)                      # ID·SUBCLASS 뺀 유전자 4,384개

# 정답 문자열(BRCA…)을 0~25 숫자로. 나중에 inverse_transform으로 되돌린다
le = LabelEncoder()
y = le.fit_transform(train['SUBCLASS'])

# 교차검증 그룹: 변이 프로필이 똑같은 환자(KIPAN/KIRC 중복 등)는 같은 fold에
groups = F.profile_hash(train, genes)

print(train.shape, '| classes', len(le.classes_), '| unique profiles', len(set(groups)))


# → train을 읽고, 정답을 숫자로 바꾸고, 중복 프로필을 묶는 그룹 id를 만들겠다.

## PART 2 · 기준선 E8 재현 (그룹 5-fold, Macro F1 ≈ 0.450)

In [ ]:
# 공통 교차검증 함수 — 팀 전원이 이 함수로만 점수를 낸다


def cv_score(make_X, label, n_estimators=100, max_depth=6, lr=0.1, verbose=True):
    """make_X(train_idx, valid_idx) -> (X_tr, X_va). fold마다 새로 fit해야 leakage가 없다."""
    skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    scores, oof = [], np.zeros(len(y), dtype=int)
    for k, (tr, va) in enumerate(skf.split(np.zeros(len(y)), y, groups)):
        X_tr, X_va = make_X(tr, va)
        model = xgboost.XGBClassifier(n_estimators=n_estimators, max_depth=max_depth, learning_rate=lr,
                                      tree_method='hist', random_state=SEED, n_jobs=-1, eval_metric='mlogloss')
        # 소수 아형을 틀리면 더 큰 벌점 — Macro F1과 학습 목표를 맞춘다 (학습 fold 라벨로만 계산)
        model.fit(X_tr, y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
        pred = model.predict(X_va); oof[va] = pred
        scores.append(f1_score(y[va], pred, average='macro'))
        if verbose: print(f'  fold {k}: {scores[-1]:.4f}')
    print(f'{label}: {np.mean(scores):.4f} ± {np.std(scores):.4f}')
    return np.mean(scores), oof


# → 5조각 그룹 교차검증을 돌려 평균 Macro F1과 OOF(각 환자를 못 본 모델이 낸 예측)를 돌려주겠다.

In [ ]:
# E8 특성 한 번만 계산 (행 단위 특성이라 미리 계산해도 leakage 없음)


binary = F.to_binary(train, genes)                 # P1: WT→0, 변이→1  (test 빈칸은 F.fill_missing이 WT로 채움)
drivers = F.load_driver_genes(genes)               # Bailey 2018 드라이버 중 우리 열에 있는 157개
eng = F.engineered(train, genes, drivers)          # 변이 수·유형 비율·치환 패턴 12종·아미노산 6그룹 전이 36×2·드라이버 2채널 (420열)
pw = F.pathway_features(binary, F.load_pathways(genes))   # Sanchez-Vega 2018 경로 10개 변이 부담 (개수·비율, 20열)
eng = pd.concat([eng, pw], axis=1)
print('engineered features:', eng.shape)

E8_COLS = [c for c in eng.columns]                 # E8 = 전부 사용

def make_X_E8(tr, va, extra=None):
    cols = F.keep_informative(binary.iloc[tr])     # P2: 학습 fold에서 변이 0인 유전자 제거 (fold 안에서 fit)
    X_tr = pd.concat([binary.iloc[tr][cols], eng.iloc[tr][E8_COLS]], axis=1)
    X_va = pd.concat([binary.iloc[va][cols], eng.iloc[va][E8_COLS]], axis=1)
    if extra is not None:                          # PART 3에서 내 파생변수를 붙이는 자리
        X_tr = pd.concat([X_tr, extra.iloc[tr]], axis=1); X_va = pd.concat([X_va, extra.iloc[va]], axis=1)
    return X_tr, X_va

base_score, base_oof = cv_score(make_X_E8, 'E8 baseline')


# → 기준선 E8을 재현해 0.45 근처(0.44~0.46)가 나오는지 확인하겠다. 다르면 환경·시드부터 의심.


## PART 3 · 내 파생변수 (여기만 수정)

In [ ]:
# 내 파생변수 — 환자 한 명(한 행) 안에서만 계산하거나, 학습 fold 통계만 쓴다


def my_features(df, genes):
    """df: train 또는 test DataFrame. 반환: df와 같은 index의 DataFrame (열 이름은 영문·숫자·_ 만)."""
    out = pd.DataFrame(index=df.index)
    # 예시 (Top8-6 긴 유전자 보정): FLAGS 목록 중 우리 열에 있는 유전자의 변이 합계
    # flags = pd.read_csv('/content/repo/refs/flags_shyr2014.csv')['gene']
    # fl = [g for g in flags if g in genes]
    # out['flags_sum'] = (df[fl] != 'WT').sum(axis=1)
    return out

extra = my_features(train, genes)
print('my features:', extra.shape, list(extra.columns)[:10])


# → 내가 맡은 파생변수를 함수 하나로 만들겠다. test에도 같은 함수를 그대로 쓰게 될 것이다.

## PART 4 · 비교

In [ ]:
# 기준선 vs 기준선 + 내 파생변수


if extra.shape[1] == 0:
    print('my_features()가 비어 있음 — PART 3을 채우세요')
else:
    new_score, new_oof = cv_score(lambda tr, va: make_X_E8(tr, va, extra), 'E8 + my features')
    print(f'변화: {new_score - base_score:+.4f}  (|변화| < 0.005 면 효과 없음으로 기록)')


# → 내 파생변수가 Macro F1을 얼마나 바꾸는지 보고, 결과를 맨 위와 logs/experiments.md에 적겠다.

In [ ]:
# 아형별 F1 — 어느 아형이 올랐고 내렸는지


oof = new_oof if extra.shape[1] else base_oof
rep = pd.DataFrame(classification_report(y, oof, target_names=le.classes_, output_dict=True)).T
rep = rep.iloc[:len(le.classes_)].sort_values('f1-score')
print(rep[['precision', 'recall', 'f1-score', 'support']].round(3).to_string())


# → 아형별 F1을 낮은 순으로 보고, 약한 아형(CESC·PAAD·KIPAN·SARC)이 움직였는지 확인하겠다.

## PART 5 · 제출 파일 (팀장만 실행 — 하루 4회 제한)

규칙: 여기서 처음 test를 읽는다. 전처리는 train으로 fit한 것을 test에 transform만.

In [ ]:
# 전체 train으로 학습 → test 예측 → 제출 파일


test = pd.read_csv(f'{DATA_DIR}/test.csv')
X_tr, X_te = F.build_features(train, test, level='E8')   # keep_informative는 train으로만 fit됨
if extra.shape[1]:
    X_tr = pd.concat([X_tr, my_features(train, genes)], axis=1)
    X_te = pd.concat([X_te, my_features(test, genes)], axis=1)

model = xgboost.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                              tree_method='hist', random_state=SEED, n_jobs=-1, eval_metric='mlogloss')
model.fit(X_tr, y, sample_weight=compute_sample_weight('balanced', y))

sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
sub['SUBCLASS'] = le.inverse_transform(model.predict(X_te))
EXP_ID, CV = 'E8', base_score                             # 제출하는 실험 ID와 CV 점수로 파일명
fname = f'{pd.Timestamp.now():%m%d}_{EXP_ID}_cv{CV:.3f}.csv'
sub.to_csv(fname, index=False, encoding='UTF-8-sig'); print(fname, sub['SUBCLASS'].value_counts().head())


# → train 전체로 학습한 모델로 test를 예측해, 실험 ID와 CV 점수를 담은 이름으로 제출 파일을 저장하겠다.